# EduRisk AI: 학생 중도 포기 위험 예측

학생의 학습 활동 데이터를 전처리한 뒤 TensorFlow DNN 이진 분류 모델로 중도 포기 위험을 예측한다.

## 1. 라이브러리 및 경로 설정

In [ ]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH = ROOT / 'data' / 'student_dropout.csv'
MODEL_DIR = ROOT / 'model'
MODEL_DIR.mkdir(exist_ok=True)

## 2. 데이터 불러오기

In [ ]:
df = pd.read_csv(DATA_PATH)
df.head()

## 3. 전처리

- `student_id` 제거
- 범주형 컬럼 One-Hot Encoding
- 수치형 컬럼 정규화
- `dropout`을 0/1 라벨로 사용

In [ ]:
categorical_features = ['gender', 'course_type', 'previous_grade']
numeric_features = [
    'age', 'attendance_rate', 'assignment_submit_rate', 'quiz_average',
    'login_count', 'video_watch_time', 'forum_activity'
]

df = df.drop(columns=['student_id'])
X = df[categorical_features + numeric_features]
y = df['dropout'].astype(int)

try:
    encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
except TypeError:
    encoder = OneHotEncoder(handle_unknown='ignore', sparse=False)

preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', StandardScaler(), numeric_features),
        ('categorical', encoder, categorical_features),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)
X_train_processed.shape

## 4. TensorFlow DNN 이진 분류 모델

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train_processed.shape[1],)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')]
)

history = model.fit(
    X_train_processed,
    y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)]
)

## 5. 평가 및 저장

In [ ]:
threshold = 0.7
probabilities = model.predict(X_test_processed).ravel()
predictions = (probabilities >= threshold).astype(int)

print('Accuracy:', accuracy_score(y_test, predictions))
print('Confusion Matrix:')
print(confusion_matrix(y_test, predictions))
print(classification_report(y_test, predictions, zero_division=0))

model.save(MODEL_DIR / 'edurisk_model.keras')
joblib.dump(preprocessor, MODEL_DIR / 'preprocessor.joblib')

metadata = {
    'threshold': threshold,
    'categorical_features': categorical_features,
    'numeric_features': numeric_features,
}
(MODEL_DIR / 'metadata.json').write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')